# Running Pyperplan on PDDL files

Run this notebook from the folder containing the `.pddl` files. Execute the cells from top to bottom. If you are using JupyterLite, upload this notebook and the PDDL files into the same folder.

In [1]:
%pip install pyperplan


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import io
import logging
import re
import time

from pyperplan.planner import HEURISTICS, SEARCHES, search_plan


def solve(domain, problem, search="gbf", heuristic="hff", show_plan=True):
    log = io.StringIO()
    logging.basicConfig(level=logging.INFO, stream=log, force=True)
    start = time.time()
    try:
        plan = search_plan(domain, problem, SEARCHES[search], HEURISTICS[heuristic])
    except (KeyError, OSError, ValueError, RuntimeError) as error:
        print(f"Error: {type(error).__name__}: {error}")
        print("Check that the filenames are correct and that the PDDL uses Pyperplan-supported STRIPS syntax.")
        return None

    seconds = time.time() - start
    found = re.findall(r"(\d+) Nodes expanded", log.getvalue())
    expanded = int(found[-1]) if found else None
    if plan is None:
        print(f"{search}/{heuristic}: no plan found ({expanded} nodes expanded, {seconds:.1f} s)")
        return None, expanded, seconds
    
    print(f"{search}/{heuristic}: plan with {len(plan)} steps, {expanded} nodes expanded, {seconds:.1f} s")
    if show_plan:
        for number, operation in enumerate(plan, 1):
            print(f"  {number:3}. {operation.name}")
    return len(plan), expanded, seconds


def compare(domain, problem, configs=("astar/blind", "astar/hmax", "astar/lmcut", "gbf/hff")):
    for config in configs:
        search, heuristic = config.split("/")
        solve(domain, problem, search, heuristic, show_plan=False)


print("searches:  ", ", ".join(sorted(SEARCHES)))
print("heuristics:", ", ".join(sorted(HEURISTICS)))

searches:   astar, bfs, ehs, gbf, ids, sat, wastar
heuristics: blind, hadd, hff, hmax, hsa, landmark, lmcut


In [3]:
solve("blocksworld-domain.pddl", "sussman.pddl", "astar", "blind")

astar/blind: plan with 6 steps, 16 nodes expanded, 0.0 s
    1. (unstack c a)
    2. (put-down c)
    3. (pick-up b)
    4. (stack b c)
    5. (pick-up a)
    6. (stack a b)


(6, 16, 0.0022017955780029297)

In [4]:
solve("Shopping-domain.pddl", "Shopping-problem.pddl", "astar", "blind")

astar/blind: plan with 6 steps, 22 nodes expanded, 0.0 s
    1. (go home hws)
    2. (buy drill hws)
    3. (go hws sm)
    4. (buy bananas sm)
    5. (buy milk sm)
    6. (go sm home)


(6, 22, 0.0018389225006103516)

## Blocks World

The 8-block problem is included in this folder. The Sussman example is optional and requires a separate `sussman.pddl` file.

In [5]:
compare("blocksworld-domain.pddl", "blocksworld-8.pddl")

astar/blind: plan with 16 steps, 75300 nodes expanded, 1.3 s
astar/hmax: plan with 16 steps, 4424 nodes expanded, 2.8 s
astar/lmcut: plan with 16 steps, 22 nodes expanded, 0.1 s
gbf/hff: plan with 16 steps, 100 nodes expanded, 0.0 s


## Assignment 7: pre-marshalling

Complete the TODOs in `premarshalling-domain.pddl` and `premarshalling-bay1.pddl` before running this cell.

In [6]:
solve("premarshalling-domain.pddl", "premarshalling-bay1.pddl", "astar", "blind")

astar/blind: plan with 0 steps, 1 nodes expanded, 0.0 s


(0, 1, 0.008360862731933594)